# S0/S1 sanity ladder: oracle query path and non-amortised shift ceiling (A→B architecture)

## Description

<!-- Written before the first run (2026-10-28); kept up to date. Design: docs/design_2026-10-28_A_to_B_architecture.md §8. Issue #16. -->

**Question.** The proposed architecture moves A's points and queries *into* B's domain and reads B through the frozen
single-task PFN (the query path $Q_B$, B's per-layer K/V cached once). Two questions are answered before anything is trained:

- **S0, plumbing.** Is the *oracle particle* $Q_B(T(x^*))$ (B queried at the true transform, A unused) as good as the
  privileged teacher, and much better than the A-only PFN? This validates the query path, caching and scale handling
  ($h = \mathrm{id}$: $Q_B$'s y-scale is A's).
- **S1, non-amortised ceiling.** With $T(x) = x + s$ unknown, how much of the oracle's value can a Bayes learner recover
  from A's few points? Posterior over $s$ on a grid, its MAP error, entropy, **fraction of unimodal posteriors** (when is one
  hypothesis enough), and whether gradient descent on $s$ through $Q_B$ finds the MAP (is the landscape navigable).

**Setup.** Frozen `pfn_large` (pfn-repro v0.1.0, the variable-d checkpoint, trained on contexts $n \le 100$), no training.
$f \sim$ BNN prior on $[0,1]^d$ (the PFN's training prior and ECDF scale), $h = \mathrm{id}$, $T(x) = x + s$,
$s \sim U(-0.2, 0.2)^d$, $d \in \{1, 2\}$. B: uniform design, $N_B = 80$ (main arm; $N_B + n_A \le 90$). A: train and query
inputs drawn so that $T(x) \in [0,1]^d$; $n_A \in \{0, 1, 2, 5, 10\}$ (nested), 50 held-out queries. 100 pairs per stratum:
`shift`; `identity` ($s = 0$, sanity); `unrelated` (B observes an independent function: negative transfer). Split 0 (pilot).
One member per $(d, N_B)$; $N_B = 200$ (the design note's value) is run as an **out-of-range probe** only.

| predictor | context | query | role |
|---|---|---|---|
| `a_only` | $A_{<n}$ | $x^*$ | lower bound |
| `oracle_q` | $B$ | $T(x^*)$ | oracle particle (S0 (b)) |
| `teacher_a` | $A_{<n} \cup B_{\mathrm{in}A}$ (B points with $x_b - s \in [0,1]^d$) | $x^*$ | A-space teacher (S0 (c)) |
| `teacher_b` | $B \cup A_{\mathrm{in}B,<n}$ | $T(x^*)$ | B-space teacher (§7.2) |
| `naive` | $B \cup A_{<n}$ | $x^*$ | pooling without registration |
| `grid_pseudo` | $B$ | $x^* + s$, mixed over the grid posterior | S1, the architecture's pseudo-likelihood $\sum_a \log Q_B(y_a \mid x_a + s)$ |
| `map_pseudo` | $B$ | $x^* + \hat s_{\mathrm{MAP}}$ | S1, one particle at the MAP (what S2 aims at) |
| `grid_exact` | $B \cup A_{\mathrm{in}B,<n}(s)$ | $x^* + s$, chain-rule posterior | S1, the exact Bayes learner for the shift model |
| `bayes_null` | mixture of `grid_exact` and `a_only`, prior 1/2 | | S1 with the null hypothesis ("unrelated") |

Grid: 201 points ($d=1$, step 0.002), $41^2$ ($d=2$, step 0.01), uniform prior (= the generating prior); $x + s$ clipped to
$[0,1]^d$. GD: Adam (lr 0.01, 200 steps) on the pseudo log-likelihood from $s = 0$, $s$ projected to the prior box; success =
its log-likelihood within 0.1 nats of the grid MAP's.

**What to look for.** (1) NLL vs $n_A$ per predictor, stratum and $d$ (mean ± SE over pairs: the MC error);
(2) $d=1$ examples: $f_A$, the oracle particle and the A-only PFN; (3) posteriors over $s$ at $n_A = 1, 2, 5, 10$;
(4) MAP error, entropy, unimodal fraction vs $n_A$; (5) GD success and trajectories on the likelihood landscape;
(6) the $N_B = 200$ probe against $N_B = 80$.

**GO / NO-GO** (*drafted by Claude, 2026-10-28, before the first run*; shift stratum, $N_B = 80$, paired over pairs):

- **S0 PASS** if for both $d$, at every $n_A \in \{0, 1, 2\}$:
  (P1) NLL(`a_only`) − NLL(`oracle_q`) ≥ 0.5 nats/query with the 95% CI's lower bound > 0 ("≪");
  (P2) |NLL(`oracle_q`) − NLL(`teacher_b`)| ≤ 0.1 and |NLL(`oracle_q`) − NLL(`teacher_a`)| ≤ 0.25 ("≈"; `teacher_a` sees
  fewer B points). S0 FAIL otherwise. Exact identities ($n_A = 0$: `teacher_b` = `oracle_q`; the first chain-rule term =
  the pseudo-likelihood) are unit tests (`tests/unit/test_ladder.py`).
- **S1 → build S2?** At $n_A = 5$: **GO** if for both $d$ `grid_pseudo` closes ≥ 50% of the gap `a_only` → `oracle_q`
  (ratio of means) *and* the unimodal fraction (pseudo posterior) is ≥ 0.6; **NO-GO** if for either $d$ it closes < 20%
  (the location signal is not in A's few points); **INCONCLUSIVE** otherwise.
- Descriptive, not gated: GD success rate (< 50% means S2 needs refinement steps or several starts), the
  pseudo-vs-exact gap (cost of the independence assumption, §6), the unrelated stratum's harm and `bayes_null`'s recovery.

**Limits.** `pfn_large` is valid for $n \le 100$, $d \le 6$: the $N_B = 200$ probe is outside and is not used for any verdict.
Shift-only $T$, $h = \mathrm{id}$, uniform B, no residual: the easiest rung by design. Wrong hypotheses push $x + s$ outside
the box, where the input is clipped (the frozen PFN never saw $x \notin [0,1]$). The grid prior equals the generating prior,
which favours the Bayes learner. Pilot split only; nothing here is tuned on split 1.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/ladder/<sweep_id>"  # set after `just snapshot runs/ladder/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.